# STRATA Demo: Download and visualize a snapshot in time

**BEFORE YOU BEGIN:** A free [Globus account](https://www.globus.org/) is required; you can sign up with an existing Google or institutional account.


This notebook demonstrates how to download and visualize a snapshot in time from the following dataset:

[SST-TG-P1F4R3200](https://doi.ccs.ornl.gov/dataset/5be73ed1-f138-504e-9851-4dff0f465a1d) — (Taylor-Green: Pr = 1, Fr = 4, Re = 3200)
- This dataset contains timestamps labelled from 1 to 15000, with each timestamp containing 4 variables (`u1`, `u2`, `u3` (velocity components) and `rho` (turbulent density fluctuations)) 
- Each snapshot contains 512 × 512 × 256 gridpoints, and data is saved in single precision format, yielding a filesize of roughly 268 MB.


The following tutorial will load the four primative variables from a timestamp of your choosing, and create an interactive visualization of the domain.

---
## Step 1: Install dependencies, import required libraries

In [25]:
!pip install globus-sdk requests tqdm -q

In [27]:
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import globus_sdk
import requests
from tqdm.auto import tqdm

---
## Step 2: Configuration

Choose which time step and variables to download. All other values are pre-filled.

In [21]:
# --- User selection ---
TIMESTEP   = 11100                       # time step index to download (1 to 15000 in steps of 1)
VARIABLES  = ["rho", "u1", "u2", "u3"]  # load all variables
OUTPUT_DIR = "./strata_data"            # where to save the files

# --- Pre-filled GLOBUS settings ---
CLIENT_ID       = "d47db6dc-0428-4076-9a6e-31927d7c7704"
COLLECTION_ID   = "57618e0a-2c99-45ff-9694-24141b92fa17"
HTTPS_BASE      = "https://g-e320e6.63720f.75bc.data.globus.org"
COLLECTION_PATH = "/gen101/world-shared/doi-data/OLCF/202504/10.13139_OLCF_2530508"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Files will be saved to: {os.path.abspath(OUTPUT_DIR)}")

Files will be saved to: /Users/mcouchman/Documents/York/Research/StratTurbWebsite/web/strata_data


---
## Step 3: Authenticate with Globus

Run the cell below, click the login URL, sign in with your Globus account, and paste the authorisation code back here.

In [5]:
HTTPS_SCOPE = f"https://auth.globus.org/scopes/{COLLECTION_ID}/https"

client = globus_sdk.NativeAppAuthClient(CLIENT_ID)
client.oauth2_start_flow(requested_scopes=HTTPS_SCOPE)

print("Open this URL in your browser to log in:")
print()
print(client.oauth2_get_authorize_url())
print()

auth_code  = input("Paste the authorisation code here: ").strip()
token_resp = client.oauth2_exchange_code_for_tokens(auth_code)

https_token = None
for rs, data in token_resp.by_resource_server.items():
    if rs != "auth.globus.org":
        https_token = data["access_token"]
        break

assert https_token, "Authentication failed: no HTTPS token received."
print("Authentication successful.")

Open this URL in your browser to log in:

https://auth.globus.org/v2/oauth2/authorize?client_id=d47db6dc-0428-4076-9a6e-31927d7c7704&redirect_uri=https%3A%2F%2Fauth.globus.org%2Fv2%2Fweb%2Fauth-code&scope=https%3A%2F%2Fauth.globus.org%2Fscopes%2F57618e0a-2c99-45ff-9694-24141b92fa17%2Fhttps&state=_default&response_type=code&code_challenge=PeLQpRnP-cBwCEEGcYR1V0m3PZuqTvT9gHd5JLQXYkY&code_challenge_method=S256&access_type=online

Authentication successful.


---
## Step 4: File download

Four variables from the timestamp you selected above will be downloaded (it may take a few minutes, you can choose a subset of variables to speed things up)

In [22]:
def timestep_subdir(t):
    idx   = (t - 1) // 1000 + 1
    start = (idx - 1) * 1000 + 1
    end   = idx * 1000
    return f"{idx:02d}_{start}to{end}"

def download_file(url, output_path, token):
    headers = {"Authorization": f"Bearer {token}"}
    resp    = requests.get(url, headers=headers, stream=True)
    resp.raise_for_status()

    total = 250 * 1024 * 1024  # ~250 MB per field file
    with open(output_path, "wb") as f, tqdm(
        total=total,
        unit="B", unit_scale=True, unit_divisor=1024,
        desc=os.path.basename(output_path)
    ) as bar:
        for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
            f.write(chunk)
            bar.update(len(chunk))


subdir           = timestep_subdir(TIMESTEP)
downloaded_files = {}

print(f"Time step {TIMESTEP} → subdirectory: {subdir}\n")

for var in VARIABLES:
    filename    = f"{var}.{TIMESTEP}"
    url         = f"{HTTPS_BASE}{COLLECTION_PATH}/{subdir}/{filename}?download=1"
    output_path = os.path.join(OUTPUT_DIR, filename)

    download_file(url, output_path, https_token)
    downloaded_files[var] = output_path

print("\nAll files downloaded.")

Time step 11100 → subdirectory: 12_11001to12000



rho.11100:   0%|          | 0.00/268M [00:00<?, ?B/s]

u1.11100:   0%|          | 0.00/268M [00:00<?, ?B/s]

u2.11100:   0%|          | 0.00/268M [00:00<?, ?B/s]

u3.11100:   0%|          | 0.00/268M [00:00<?, ?B/s]


All files downloaded.


---
## Step 5: Load the data

The x-dimension is padded with two zeros, which are removed prior to analysis.

In [23]:
Nx, Ny, Nz = 512, 512, 256

def load_field(path):
    X = np.memmap(path, dtype="single", mode="r",shape=(Nx+2,Ny,Nz), order="F")
    return X[:-2,:,:] #Chop off two rows of zeros

fields = {var: load_field(path) for var, path in downloaded_files.items()}

for var, arr in fields.items():
    print(f"Loaded  {var}:  shape {arr.shape},  min {arr.min():.4f},  max {arr.max():.4f}")

Loaded  rho:  shape (512, 512, 256),  min -1.2823,  max 1.5255
Loaded  u1:  shape (512, 512, 256),  min -1.0941,  max 0.9827
Loaded  u2:  shape (512, 512, 256),  min -0.9672,  max 1.1623
Loaded  u3:  shape (512, 512, 256),  min -0.7900,  max 0.9004


---
The fields are now loaded as NumPy arrays in the `fields` dictionary and ready for analysis.

---
## Step 6: Interactive slice viewer

Use the dropdowns and slider to slice through any of the three axes for a specified variable.

In [ ]:
Nx, Ny, Nz = 512, 512, 256
axis_sizes = {'x': Nx, 'y': Ny, 'z': Nz,}

var_widget = widgets.Dropdown(
    options=list(fields.keys()),
    value=list(fields.keys())[0],
    description='Variable:',
    style={'description_width': 'initial'}
)

axis_widget = widgets.Dropdown(
    options=['x', 'y', 'z'],
    value='z',
    description='Slice axis:',
    style={'description_width': 'initial'}
)

slider = widgets.IntSlider(
    value=0, min=0, max=Nz - 1, step=1,
    description='Index:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

def update_slider_range(change):
    new_max = axis_sizes[axis_widget.value] - 1
    slider.max = new_max
    slider.value = min(slider.value, new_max)

axis_widget.observe(update_slider_range, names='value')

def show_slice(variable, axis, index):
    data = fields[variable]
    if axis == 'z':
        img = data[:, :, index].T;  xlabel, ylabel = 'x', 'y';  asp = 1
    elif axis == 'y':
        img = data[:, index, :].T;  xlabel, ylabel = 'x', 'z';  asp = 1
    else:
        img = data[index, :, :].T;  xlabel, ylabel = 'y', 'z';  asp = 1

    vmax = np.percentile(np.abs(data), 99)
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(img, cmap='RdBu_r', origin='lower',
                   vmin=-vmax, vmax=vmax, aspect=asp)
    plt.colorbar(im, ax=ax, label=variable, shrink=0.85)
    ax.set_xlabel(xlabel);  ax.set_ylabel(ylabel)
    ax.set_title(f'{variable}  —  {axis}-slice  {index}')
    plt.tight_layout()
    plt.show()

out = widgets.interactive_output(
    show_slice,
    {'variable': var_widget, 'axis': axis_widget, 'index': slider}
)

display(widgets.VBox([var_widget, axis_widget, slider, out]))